In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt
from tensorflow.keras.datasets import california_housing
(x_train,y_train),(x_test,y_test) = california_housing.load_data()

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [2]:
import keras_tuner as kt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import *

def build_model(hp):

    model = keras.Sequential()

    # INPUT
    model.add(Input(shape=(8,)))

    # NUMBER OF HIDDEN BLOCKS
    for i in range(hp.Int("num_layers", 1, 6)):

        # DENSE
        model.add(
            Dense(
                units=hp.Int(
                    f'units_{i}',
                    min_value=32,
                    max_value=256,
                    step=32
                )
            )
        )

        # BATCH NORM CHOICE
        if hp.Boolean(f'use_bn_{i}'):

            model.add(BatchNormalization())

        # ACTIVATION FUNCTION CHOICE
        activation = hp.Choice(
            f'activation_{i}',
            values=['relu', 'tanh', 'sigmoid']
        )

        model.add(
            Activation(activation)
        )

        # DROPOUT
        model.add(
            Dropout(
                hp.Float(
                    f'dropout_{i}',
                    min_value=0.0,
                    max_value=0.5,
                    step=0.1
                )
            )
        )

    # OUTPUT
    model.add(Dense(1))

    # LEARNING RATE
    lr = hp.Choice(
        'learning_rate',
        values=[1e-2, 1e-3, 1e-4]
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=lr
        ),
        loss='mse',
        metrics=[tf.keras.metrics.R2Score()]
    )

    return model

In [3]:
tuner=kt.RandomSearch(build_model,objective=kt.Objective("val_r2_score",direction="max"),max_trials=20)


Reloading Tuner from ./untitled_project/tuner0.json


In [4]:
tuner.search(x_train,y_train,epochs=10,validation_split=0.2)

In [5]:
best_hp = tuner.get_best_hyperparameters()[0]

print(best_hp.values)

{'num_layers': 6, 'units_0': 32, 'use_bn_0': False, 'activation_0': 'tanh', 'dropout_0': 0.2, 'learning_rate': 0.001, 'units_1': 32, 'use_bn_1': False, 'activation_1': 'relu', 'dropout_1': 0.0, 'units_2': 32, 'use_bn_2': False, 'activation_2': 'relu', 'dropout_2': 0.0, 'units_3': 32, 'use_bn_3': False, 'activation_3': 'relu', 'dropout_3': 0.0, 'units_4': 32, 'use_bn_4': False, 'activation_4': 'relu', 'dropout_4': 0.0, 'units_5': 32, 'use_bn_5': False, 'activation_5': 'relu', 'dropout_5': 0.0}


In [6]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3
)

In [7]:
best_model = tuner.hypermodel.build(best_hp)

In [8]:
history = best_model.fit(
    x_train,
    y_train,

    validation_split=0.2,

    epochs=100,

    batch_size=32,

    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

Epoch 1/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 27575207936.0000 - r2_score: -1.0798 - val_loss: 6447821824.0000 - val_r2_score: 0.5248 - learning_rate: 0.0010
Epoch 2/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 6358132224.0000 - r2_score: 0.5205 - val_loss: 4935095808.0000 - val_r2_score: 0.6363 - learning_rate: 0.0010
Epoch 3/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 5378556928.0000 - r2_score: 0.5943 - val_loss: 4459426816.0000 - val_r2_score: 0.6713 - learning_rate: 0.0010
Epoch 4/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 4883146752.0000 - r2_score: 0.6317 - val_loss: 4123671040.0000 - val_r2_score: 0.6961 - learning_rate: 0.0010
Epoch 5/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 4688540160.0000 - r2_score: 0.6464 - val_loss: 4014427136.0000 - val_r2_score: 0.7041 - learning_rate: 0.0010
Epoch 6/100
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 4592225792.0000 - r2_score: 0.6536 - val_loss: 3957094144.0000 - val_r2_score: 0.7

In [9]:
from sklearn.metrics import r2_score
y_pred=best_model.predict(x_test)
print(r2_score(y_test,y_pred))

129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
0.737773060798645
